In [2]:
!pip install scipy

   ---------------------------------------- 0.0/41.3 MB ? eta -:--:--
   - -------------------------------------- 1.8/41.3 MB 10.0 MB/s eta 0:00:04
   ------ --------------------------------- 6.3/41.3 MB 18.3 MB/s eta 0:00:02
   ------- -------------------------------- 7.9/41.3 MB 13.9 MB/s eta 0:00:03
   -------- ------------------------------- 9.2/41.3 MB 12.1 MB/s eta 0:00:03
   ---------- ----------------------------- 10.7/41.3 MB 11.0 MB/s eta 0:00:03
   ----------- ---------------------------- 12.1/41.3 MB 10.1 MB/s eta 0:00:03
   ------------- -------------------------- 14.2/41.3 MB 10.1 MB/s eta 0:00:03
   --------------- ------------------------ 15.7/41.3 MB 9.8 MB/s eta 0:00:03
   ---------------- ----------------------- 17.3/41.3 MB 9.6 MB/s eta 0:00:03
   ------------------ --------------------- 19.1/41.3 MB 9.3 MB/s eta 0:00:03
   -------------------- ------------------- 20.7/41.3 MB 9.1 MB/s eta 0:00:03
   --------------------- ------------------ 22.3/41.3 MB 9.0 MB/s eta

分析1：支付工具的便捷性，是否能显著提升用户的购买力：

问题：信用卡组的平均客单价（AOV）真的比现金组（Boleto）高吗？

T检验：两种支付方式均值的差额，是真实存在的商业趋势，还是恰好这几天信用卡组多卖了几个大件导致的偶然现象？

P值：如果我们把这个结论推广到全平台，出错的概率大吗？

In [7]:
import pandas as pd
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

# 1. 加载支付数据集
df_payments = pd.read_csv('olist_order_payments_dataset.csv')

# 2. 准备 A/B 组数据
# A组 (对照组): 使用 boleto (票据/现金类) 支付的金额
group_A = df_payments[df_payments['payment_type'] == 'boleto']['payment_value']

# B组 (实验组): 使用 credit_card (信用卡) 支付的金额
group_B = df_payments[df_payments['payment_type'] == 'credit_card']['payment_value']

# 3. 基础描述统计
print(f"--- 描述性统计 ---")
print(f"Boleto (A组) 平均金额: {group_A.mean():.2f}, 样本量: {len(group_A)}")
print(f"Credit Card (B组) 平均金额: {group_B.mean():.2f}, 样本量: {len(group_B)}")

# 4. 执行独立样本 T检验
# 因为两组样本量和方差可能不同，设置 equal_var=False
t_stat, p_val = stats.ttest_ind(group_B, group_A, equal_var=False)

print(f"\n--- 统计检验结果 ---")
print(f"T-statistic: {t_stat}")
print(f"P-value: {p_val:.6f}")

alpha = 0.05
if p_val < alpha:
    print("\n结论: P值小于0.05，拒绝原假设。支付方式对订单金额有显著影响！")
else:
    print("\n结论: P值大于0.05，无法拒绝原假设。两组差异不具备统计学显著性。")

--- 描述性统计 ---
Boleto (A组) 平均金额: 145.03, 样本量: 19784
Credit Card (B组) 平均金额: 163.32, 样本量: 76795

--- 统计检验结果 ---
T-statistic: 10.648953531265184
P-value: 0.000000

结论: P值小于0.05，拒绝原假设。支付方式对订单金额有显著影响！


信用卡均值高出了约18.29元。这初步证实了信用卡用户是更“优质”的消费群体。

T 值高达 10.64。这证明了这种差距极其稳固，不是随机波动的“噪声”。

P 值为 0.0000。这验证了结论具有“统计学显著性”，即这种差异在全量用户中几乎百分之百是真实存在的。


分析2：当物流总时长跨越某个特定天数（从 5 天到 10 天）时，用户对服务质量的感知是否会发生断崖式下跌？
从用户角度考虑运输时间=送达时间-下单时间

In [16]:
import pandas as pd
from scipy import stats

df_orders = pd.read_csv('olist_orders_dataset.csv')
df_review = pd.read_csv('olist_order_reviews_dataset.csv')

# 关联表
df_combined = pd.merge(df_orders, df_review, on='order_id', how='inner')

#转时间格式
df_combined['order_delivered_customer_date'] = pd.to_datetime(df_combined['order_delivered_customer_date'])
df_combined['order_purchase_timestamp'] = pd.to_datetime(df_combined['order_purchase_timestamp'])

#计算运输天数 (提取纯数字天数)
df_combined['shipping_days'] = (df_combined['order_delivered_customer_date'] - df_combined['order_purchase_timestamp']).dt.days

#清理空值 (没送到或没评分的单不算)
df_analysis = df_combined.dropna(subset=['shipping_days', 'review_score'])

#分组
group_A = df_analysis[df_analysis['shipping_days'] <= 5]['review_score']
group_B = df_analysis[df_analysis['shipping_days'] >= 10]['review_score']

print(f"--- 描述性统计 ---")
print(f"A组 (物流快) 平均得分: {group_A.mean():.4f}, 样本量: {len(group_A)}")
print(f"B组 (物流慢) 平均得分: {group_B.mean():.4f}, 样本量: {len(group_B)}")

t_stat, p_val = stats.ttest_ind(group_B, group_A, equal_var=False)
t_stat=abs(t_stat)
print(f"\n--- 统计检验结果 ---")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.6f}")

--- 描述性统计 ---
A组 (物流快) 平均得分: 4.4308, 样本量: 19229
B组 (物流慢) 平均得分: 3.9402, 样本量: 49889

--- 统计检验结果 ---
T-statistic: 49.5382
P-value: 0.000000


In [ ]:
这说明“物流速度”对评分的影响，比“支付方式”对金额的影响要剧烈得多

分析3：得分差取决于商家发货慢还是快递员送货慢
商家发货：发货时间-下单时间
快递员送货：送达时间-发货时间

In [17]:
#快递员送货：送达时间-发货时间
import pandas as pd
from scipy import stats

df_orders = pd.read_csv('olist_orders_dataset.csv')
df_review = pd.read_csv('olist_order_reviews_dataset.csv')

# 关联表
df_combined = pd.merge(df_orders, df_review, on='order_id', how='inner')

#转时间格式
df_combined['order_delivered_customer_date'] = pd.to_datetime(df_combined['order_delivered_customer_date'])
df_combined['order_delivered_carrier_date'] = pd.to_datetime(df_combined['order_delivered_carrier_date'])

#计算运输天数 (提取纯数字天数) 改用order_delivered_carrier_date揽收时间
df_combined['shipping_days'] = (df_combined['order_delivered_customer_date'] - df_combined['order_delivered_carrier_date']).dt.days

#清理空值 (没送到或没评分的单不算)
df_analysis = df_combined.dropna(subset=['shipping_days', 'review_score'])

#分组
group_A = df_analysis[df_analysis['shipping_days'] <= 5]['review_score']
group_B = df_analysis[df_analysis['shipping_days'] >= 10]['review_score']

print(f"--- 描述性统计 ---")
print(f"A组 (物流快) 平均得分: {group_A.mean():.4f}, 样本量: {len(group_A)}")
print(f"B组 (物流慢) 平均得分: {group_B.mean():.4f}, 样本量: {len(group_B)}")

t_stat, p_val = stats.ttest_ind(group_B, group_A, equal_var=False)
t_stat=abs(t_stat)
print(f"\n--- 统计检验结果 ---")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.6f}")

--- 描述性统计 ---
A组 (物流快) 平均得分: 4.3493, 样本量: 37335
B组 (物流慢) 平均得分: 3.8104, 样本量: 31447

--- 统计检验结果 ---
T-statistic: 52.9907
P-value: 0.000000


In [18]:
#商家发货：发货时间-下单时间
import pandas as pd
from scipy import stats

df_orders = pd.read_csv('olist_orders_dataset.csv')
df_review = pd.read_csv('olist_order_reviews_dataset.csv')

# 关联表
df_combined = pd.merge(df_orders, df_review, on='order_id', how='inner')

#转时间格式
df_combined['order_delivered_carrier_date'] = pd.to_datetime(df_combined['order_delivered_carrier_date'])
df_combined['order_purchase_timestamp'] = pd.to_datetime(df_combined['order_purchase_timestamp'])

#计算运输天数 (提取纯数字天数) 改用order_delivered_carrier_date，order_purchase_timestamp卖家发货时间
df_combined['shipping_days'] = (df_combined['order_delivered_carrier_date'] - df_combined['order_purchase_timestamp']).dt.days

#清理空值 (没送到或没评分的单不算)
df_analysis = df_combined.dropna(subset=['shipping_days', 'review_score'])

#分组
group_A = df_analysis[df_analysis['shipping_days'] <= 5]['review_score']
group_B = df_analysis[df_analysis['shipping_days'] >= 10]['review_score']

print(f"--- 描述性统计 ---")
print(f"A组 (物流快) 平均得分: {group_A.mean():.4f}, 样本量: {len(group_A)}")
print(f"B组 (物流慢) 平均得分: {group_B.mean():.4f}, 样本量: {len(group_B)}")

t_stat, p_val = stats.ttest_ind(group_B, group_A, equal_var=False)
t_stat=abs(t_stat)
print(f"\n--- 统计检验结果 ---")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.6f}")

--- 描述性统计 ---
A组 (物流快) 平均得分: 4.1865, 样本量: 85731
B组 (物流慢) 平均得分: 3.3496, 样本量: 3802

--- 统计检验结果 ---
T-statistic: 31.2628
P-value: 0.000000


【A/B测试业务结论：物流全链路对用户评价的影响】

1. 谁的杀伤力最大？（看均值差：单点打击）

结论：卖家发货极其缓慢是导致极端低分的“核弹”。

证据：下单-发货 超过10天时，评分跌至全场最低 3.350 (跌幅达 0.84)。

建议：必须针对卖家设置“发货超时红线”，防止产生毁灭性的品牌差评。


2. 谁的影响面最广？（看 T值：系统稳定性） 
结论：快递员运送慢是拖累平台大盘分数的“元凶”。

证据：发货-送达 的 T值(52.99) 全场最高，样本量(3.1w) 极大。

结论：用户对运输延迟的反应极其一致且普遍。优化物流商效率是保住 4.0 分基本盘的关键。